# LeetCode 3267 — Count Almost Equal Pairs II bằng Python

- Link: https://leetcode.com/problems/count-almost-equal-pairs-ii/
- Độ khó: **Hard**
- Chủ đề: **Chuẩn hóa độ dài** (đệm số 0 bên trái), **liệt kê mọi xâu đạt được bằng ≤ 2 phép đổi chỗ** (≤ ~460 xâu), đếm cặp bằng **bảng băm** các số đã gặp

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def countPairs(self, nums: List[int]) -> int:
```

## 1. Mục tiêu học

1. Hiểu "gần bằng": được thực hiện **tổng cộng tối đa 2** phép đổi chỗ hai chữ số (mỗi phép chọn `x` hoặc `y`), cho phép số 0 đứng đầu sau khi đổi.
2. Chứng minh: một phép trên `y` tương đương một phép trên `x` (phép đổi chỗ là tự nghịch đảo) ⇒ chỉ cần xét **2 phép trên một số**.
3. Đệm mọi số về cùng độ dài `D` (≤ 7 chữ số với `nums[i] < 10^7`) ⇒ so sánh chuỗi.
4. Với mỗi số, sinh tập `reach(x)` (≤ 197 xâu phân biệt), cộng số lần các số **trước đó** nằm trong tập ⇒ `O(n · D⁴)`.
5. Kiểm chứng bằng vét cạn theo đúng định nghĩa (đổi chỗ trên chính chữ số của `x` hoặc `y`).

## 2. Đề bài & ví dụ

Hai số nguyên `x`, `y` **gần bằng** nếu có thể làm chúng bằng nhau sau **tối đa hai** lần thao tác: chọn `x` hoặc `y` và đổi chỗ hai chữ số bất kỳ của số đó (cho phép số 0 đứng đầu). Đếm số cặp chỉ số `i < j` mà `nums[i]`, `nums[j]` gần bằng.

**Ví dụ 1:** `[1023, 2310, 2130, 213]` → `4`. **Ví dụ 2:** `[1, 10, 100]` → `3`.

**Ràng buộc:** `2 ≤ n ≤ 5000`; `1 ≤ nums[i] < 10^7`.

In [1]:
from typing import List
from collections import Counter

examples = [([1023, 2310, 2130, 213], 4), ([1, 10, 100], 3)]
for nums, exp in examples:
    print(nums, '-> kỳ vọng', exp)

[1023, 2310, 2130, 213] -> kỳ vọng 4
[1, 10, 100] -> kỳ vọng 3


## 3. Kiến thức nền

### 3.1. Sinh mọi xâu đạt được bằng ≤ 2 phép đổi chỗ

In [2]:
def reach(s):
    s = list(s); L = len(s)
    out = {''.join(s)}
    for i in range(L):
        for j in range(i + 1, L):
            s[i], s[j] = s[j], s[i]
            out.add(''.join(s))
            for a in range(L):
                for b in range(a + 1, L):
                    s[a], s[b] = s[b], s[a]
                    out.add(''.join(s))
                    s[a], s[b] = s[b], s[a]
            s[i], s[j] = s[j], s[i]
    return out

print('|reach("1023")| =', len(reach("1023")), '| "2310" có trong đó:', "2310" in reach("1023"))
print('|reach("1234567")| =', len(reach("1234567")), '= 1 + 21 + 175 (khoảng cách 0, 1, 2; 175 là số Stirling loại một c(7, 5))')

|reach("1023")| = 18 | "2310" có trong đó: True
|reach("1234567")| = 197 = 1 + 21 + 175 (khoảng cách 0, 1, 2; 175 là số Stirling loại một c(7, 5))


### 3.2. Đệm số 0 bên trái

`213` và `1023` có độ dài khác nhau; đệm thành `0213` và `1023`. Phép đổi chỗ trên `1023` (số dài hơn) đưa số 0 lên đầu tương ứng với phép so sánh trên xâu đã đệm. Vì phép đổi chỗ là tự nghịch đảo, quan hệ "gần bằng" trên xâu đã đệm là **đối xứng**.

## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Ví dụ 2: `1, 10, 100` đệm thành `001, 010, 100`. Mỗi cặp khác nhau bởi **một** phép đổi chỗ ⇒ 3 cặp ✓.

Ví dụ 1: `1023 → 2310` cần 2 phép (`1↔2` rồi `0↔3`), `2310 ↔ 2130` cần 1 phép, `1023 ↔ 0213` cần 2 phép, `2310 ↔ 0213` cần 2 phép; `2130 ↔ 1023`? `2130 → 1023` cần 3 phép (hoán vị gồm chu trình dài 4) ⇒ không. Tổng 4 ✓.

## 5. Cách 1 — Brute Force: tìm kiếm theo đúng định nghĩa cho từng cặp

In [3]:
def almost_equal_def(x, y):
    """BFS trên (xâu của x, xâu của y) với tối đa 2 phép, mỗi phép đổi chỗ trong x hoặc y."""
    frontier = {(str(x), str(y))}
    seen = set(frontier)
    for step in range(3):
        for sx, sy in frontier:
            if int(sx) == int(sy):
                return True
        if step == 2:
            break
        nxt = set()
        for sx, sy in frontier:
            for which in (0, 1):
                s = list(sx if which == 0 else sy)
                for i in range(len(s)):
                    for j in range(i + 1, len(s)):
                        t = s[:]; t[i], t[j] = t[j], t[i]
                        st = (''.join(t), sy) if which == 0 else (sx, ''.join(t))
                        if st not in seen:
                            seen.add(st); nxt.add(st)
        frontier = nxt
    return False

def brute(nums):
    n = len(nums)
    return sum(1 for i in range(n) for j in range(i + 1, n) if almost_equal_def(nums[i], nums[j]))

for nums, exp in examples:
    print(brute(nums), '| kỳ vọng', exp)

4 | kỳ vọng 4
3 | kỳ vọng 3


**Độ phức tạp:** rất lớn mỗi cặp (`O(D⁸)` trạng thái) — chỉ cho dữ liệu nhỏ, nhưng **đúng theo định nghĩa** (không giả định gì).

## 6. Ý tưởng thuật toán / bước đột phá

```
D = số chữ số lớn nhất; đệm mọi số về D chữ số
cnt = Counter()
for x in nums (theo thứ tự bất kỳ):
    s = pad(x)
    ans += Σ_{t ∈ reach(s)} cnt[t]
    cnt[s] += 1
```

Mỗi cặp `(i, j)` được đếm đúng một lần — khi xử lý phần tử đến sau. `|reach| ≤ 197` xâu phân biệt (với 7 chữ số khác nhau), sinh bằng `≤ 21²` phép thử ⇒ `≈ 10^6` phép tra cứu.

## 7. Các cách tiếp theo

### 7.1. Cách 2 — So sánh từng cặp bằng vị trí sai khác (`O(n² D)`)

Hai xâu đã đệm `a`, `b` cùng độ dài: gần bằng ⇔ cùng đa tập chữ số và hoán vị biến `a` thành `b` cần ≤ 2 phép đổi chỗ. Cụ thể với tập vị trí sai `P`: `|P| = 0` (bằng nhau), `|P| = 2` (một phép), `|P| = 3` (chu trình độ dài 3 — 2 phép), `|P| = 4` (hai cặp đổi chỗ rời nhau, **hoặc** một chu trình 4 — cần 3 phép ⇒ loại). Với chữ số lặp, có thể chọn cách ghép tốt nhất — ta kiểm tra bằng thử ≤ 2 phép trên các vị trí sai.

In [4]:
from itertools import combinations
def pair_check(a, b):
    if a == b:
        return True
    P = [i for i in range(len(a)) if a[i] != b[i]]
    if len(P) > 4 or sorted(a[i] for i in P) != sorted(b[i] for i in P):
        return False
    s = list(a)
    for i, j in combinations(P, 2):
        s[i], s[j] = s[j], s[i]
        if ''.join(s) == b:
            return True
        for u, v in combinations(P, 2):
            s[u], s[v] = s[v], s[u]
            if ''.join(s) == b:
                return True
            s[u], s[v] = s[v], s[u]
        s[i], s[j] = s[j], s[i]
    return False

def pairwise_solution(nums):
    D = len(str(max(nums)))
    S = [str(x).zfill(D) for x in nums]
    n = len(S)
    return sum(1 for i in range(n) for j in range(i + 1, n) if pair_check(S[i], S[j]))

for nums, exp in examples:
    print(pairwise_solution(nums), '| kỳ vọng', exp)

4 | kỳ vọng 4
3 | kỳ vọng 3


Chỉ cần xét phép đổi chỗ **trong các vị trí sai**: đổi chỗ đụng tới một vị trí đã đúng sẽ phải đổi lại, tốn thêm phép.

### 7.2. Cách 3 — Tập đạt được + bảng đếm (lời giải chính)

In [5]:
def reach_counter(nums):
    D = len(str(max(nums)))
    cnt = Counter()
    ans = 0
    for x in nums:
        s = str(x).zfill(D)
        for t in reach(s):
            ans += cnt[t]
        cnt[s] += 1
    return ans

for nums, exp in examples:
    print(reach_counter(nums), '| kỳ vọng', exp)

4 | kỳ vọng 4
3 | kỳ vọng 3


### 7.3. Cách 4 — Gom theo giá trị trùng lặp trước

Nhiều số giống hệt nhau? Đếm tần suất `f[s]` của mỗi xâu đã đệm; cặp trong cùng nhóm: `C(f, 2)`; cặp giữa hai nhóm khác nhau `s < t` (gần bằng): `f[s] · f[t]`. Sinh `reach` chỉ cho các xâu **phân biệt**.

In [6]:
def grouped(nums):
    D = len(str(max(nums)))
    f = Counter(str(x).zfill(D) for x in nums)
    ans = sum(c * (c - 1) // 2 for c in f.values())
    for s, c in f.items():
        for t in reach(s):
            if t != s and t in f and s < t:
                ans += c * f[t]
    return ans

for nums, exp in examples:
    print(grouped(nums), '| kỳ vọng', exp)

4 | kỳ vọng 4
3 | kỳ vọng 3


Điều kiện `s < t` để mỗi cặp nhóm được đếm một lần (quan hệ đối xứng: `t ∈ reach(s)` ⇔ `s ∈ reach(t)`).

## 8. TRACE từng bước

Trace cách 3 trên ví dụ 1.

In [7]:
def trace(nums):
    D = len(str(max(nums)))
    cnt = Counter(); ans = 0
    for x in nums:
        s = str(x).zfill(D)
        hits = {t: cnt[t] for t in reach(s) if cnt[t]}
        ans += sum(hits.values())
        print(f"x = {x} -> '{s}': khớp với các số trước {hits} -> ans = {ans}")
        cnt[s] += 1
    print('đáp án:', ans, '| vét cạn:', brute(nums))

trace(examples[0][0])

x = 1023 -> '1023': khớp với các số trước {} -> ans = 0
x = 2310 -> '2310': khớp với các số trước {'1023': 1} -> ans = 1
x = 2130 -> '2130': khớp với các số trước {'2310': 1} -> ans = 2
x = 213 -> '0213': khớp với các số trước {'1023': 1, '2310': 1} -> ans = 4
đáp án: 4 | vét cạn: 4


## 9. Code nộp LeetCode

In [8]:
class Solution:
    def countPairs(self, nums: List[int]) -> int:
        D = len(str(max(nums)))
        cnt = {}
        ans = 0
        for x in nums:
            s = list(str(x).zfill(D))
            seen = {''.join(s)}
            for i in range(D):
                for j in range(i + 1, D):
                    if s[i] == s[j]:
                        continue
                    s[i], s[j] = s[j], s[i]
                    seen.add(''.join(s))
                    for a in range(i + 1, D):
                        for b in range(a + 1, D):
                            if s[a] != s[b]:
                                s[a], s[b] = s[b], s[a]
                                seen.add(''.join(s))
                                s[a], s[b] = s[b], s[a]
                    s[i], s[j] = s[j], s[i]
            for t in seen:
                ans += cnt.get(t, 0)
            key = ''.join(s)
            cnt[key] = cnt.get(key, 0) + 1
        return ans

for nums, exp in examples:
    got = Solution().countPairs(nums)
    print(got, 'OK' if got == exp else 'SAI')

4 OK
3 OK


Tối ưu nhỏ (đã kiểm chứng ở mục 12 bằng đối chiếu với `reach` đầy đủ): bỏ qua đổi chỗ hai chữ số **giống nhau** (không đổi xâu), và phép thứ hai chỉ xét `a > i`. Lý do: tích hai phép đổi chỗ hoặc là hai phép **rời nhau** (giao hoán được — đặt phép có chỉ số nhỏ nhất làm phép đầu), hoặc là một **chu trình 3** trên `{p < q < r}` — chu trình này luôn viết được thành "đổi một cặp chứa `p` trước, rồi đổi `(q, r)`", tức phép thứ hai có chỉ số nhỏ nhất `q > p = i`.

## 10. Vì sao thuật toán đúng

1. **Chỉ cần thao tác trên một số:** nếu áp dụng phép `σ` lên `x` và `τ` lên `y` để `σ(x) = τ(y)`, thì `τ(σ(x)) = y` (vì `τ` tự nghịch đảo) — hai phép trên `x`. Tương tự hai phép trên `y` ⇔ hai phép (nghịch đảo, theo thứ tự ngược) trên `x`.
2. **Đệm số 0:** bằng nhau về giá trị ⇔ bằng nhau khi đệm về cùng độ dài; đổi chỗ trên số dài hơn có thể đưa số 0 lên đầu, khớp với xâu đệm của số ngắn hơn. (Phép đổi chỗ trên xâu đệm của số ngắn có dùng vị trí đệm tương ứng với phép ngược trên số dài.)
3. **Đếm:** mỗi cặp `(i, j)` với `i < j` được cộng đúng một lần ở bước `j`, vì `pad(nums[i]) ∈ reach(pad(nums[j]))` ⇔ gần bằng.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. BFS theo định nghĩa | `O(n² D⁸)` | lớn |
| 2. So từng cặp | `O(n² D)` (+ hằng số nhỏ) | `O(n D)` |
| 3. Tập đạt được + bảng | `O(n D⁴)` | `O(n D + D⁴)` |
| 4. Gom nhóm | `O(u D⁴ + n)` (`u` số phân biệt) | `O(u D)` |

## 12. Bộ test

### 12.1. Ví dụ đề + ca biên

In [9]:
impls = {'pairwise': pairwise_solution, 'reach_counter': reach_counter, 'grouped': grouped,
         'submit': lambda a: Solution().countPairs(a)}
edge = [([5, 5], 1), ([12, 21], 1), ([123, 231], 1), ([1234, 2143], 1), ([1234, 4123], 0), ([1, 1, 1], 3), ([100, 1], 1)]
for nums, exp in examples + edge:
    assert brute(nums) == exp, (nums, brute(nums))
    for name, f in impls.items():
        assert f(nums) == exp, (name, nums)
print('Ví dụ đề và ca biên: tất cả đúng.')

Ví dụ đề và ca biên: tất cả đúng.


`1234 → 4123` là chu trình độ dài 4 — cần 3 phép đổi chỗ ⇒ không gần bằng.

### 12.2. Đối chiếu ngẫu nhiên với định nghĩa

In [10]:
import random
rng = random.Random(3267)
cnt_t = 0
for _ in range(120):
    n = rng.randint(2, 6)
    digits = rng.choice(['012', '0123', '1123', '01'])
    nums = [int(''.join(rng.choice(digits) for _ in range(rng.randint(1, 4)))) or 1 for _ in range(n)]
    exp = brute(nums)
    for name, f in impls.items():
        assert f(nums) == exp, (name, nums)
    cnt_t += 1
print('Khớp định nghĩa (BFS) trên', cnt_t, 'mảng ngẫu nhiên nhiều chữ số lặp.')

Khớp định nghĩa (BFS) trên 120 mảng ngẫu nhiên nhiều chữ số lặp.


### 12.3. Ca lớn `n = 5000`

In [11]:
import time
big = {
    'ngẫu nhiên < 1e7': [rng.randint(1, 10 ** 7 - 1) for _ in range(5000)],
    'hoán vị của 1234567': [int(''.join(rng.sample('1234567', 7))) for _ in range(5000)],
    'chữ số 0/1': [int('1' + ''.join(rng.choice('01') for _ in range(6))) for _ in range(5000)],
}
for name, nums in big.items():
    t = time.time(); a = Solution().countPairs(nums); t1 = time.time() - t
    t = time.time(); b = grouped(nums); t2 = time.time() - t
    assert a == b
    print(f"{name:22s}: {a:9d} | nộp {t1:.2f}s | gom nhóm {t2:.2f}s")

ngẫu nhiên < 1e7      :       200 | nộp 0.30s | gom nhóm 0.62s


hoán vị của 1234567   :    488586 | nộp 0.39s | gom nhóm 0.42s
chữ số 0/1            :   2752260 | nộp 0.12s | gom nhóm 0.01s


Kiểm tra độc lập cho ca nhỏ hơn: so cách 2 (so từng cặp, không sinh tập) với lời giải nộp trên `n = 800`.

In [12]:
sub = big['hoán vị của 1234567'][:800]
assert pairwise_solution(sub) == Solution().countPairs(sub)
sub2 = big['chữ số 0/1'][:800]
assert pairwise_solution(sub2) == Solution().countPairs(sub2)
print('so từng cặp = lời giải nộp trên 800 phần tử (hai bộ).')

so từng cặp = lời giải nộp trên 800 phần tử (hai bộ).


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Chỉ cho phép **một** phép đổi chỗ (bản I)

In [13]:
def wrong_one_swap(nums):
    D = len(str(max(nums))); cnt = Counter(); ans = 0
    for x in nums:
        s = list(str(x).zfill(D)); out = {''.join(s)}
        for i in range(D):
            for j in range(i + 1, D):
                s[i], s[j] = s[j], s[i]; out.add(''.join(s)); s[i], s[j] = s[j], s[i]
        ans += sum(cnt[t] for t in out)
        cnt[''.join(s)] += 1
    return ans

print('sai :', wrong_one_swap(examples[0][0]), '| đúng:', Solution().countPairs(examples[0][0]))

sai : 1 | đúng: 4


**Vì sao sai:** bản II cho phép **hai** phép — `1023 ↔ 2310` cần đúng hai.

### Lỗi 2 — Không đệm số 0 (so chuỗi khác độ dài)

In [14]:
def wrong_no_pad(nums):
    cnt = Counter(); ans = 0
    for x in nums:
        s = str(x)
        ans += sum(cnt[t] for t in reach(s))
        cnt[s] += 1
    return ans

print('sai :', wrong_no_pad(examples[1][0]), '| đúng:', Solution().countPairs(examples[1][0]))

sai : 0 | đúng: 3


**Vì sao sai:** `10` đổi chỗ thành `01` = `1`, nhưng xâu `"01" ≠ "1"`. Phải so theo giá trị — tương đương đệm về cùng độ dài.

### Lỗi 3 — Sinh các xâu đạt được thành **danh sách** (đếm lặp)

In [15]:
def wrong_list(nums):
    D = len(str(max(nums))); cnt = Counter(); ans = 0
    for x in nums:
        s = list(str(x).zfill(D)); out = [''.join(s)]
        for i in range(D):
            for j in range(i + 1, D):
                s[i], s[j] = s[j], s[i]; out.append(''.join(s))
                for a in range(D):
                    for b in range(a + 1, D):
                        s[a], s[b] = s[b], s[a]; out.append(''.join(s)); s[a], s[b] = s[b], s[a]
                s[i], s[j] = s[j], s[i]
        ans += sum(cnt[t] for t in out)                         # SAI: một xâu đạt được nhiều cách
        cnt[''.join(s)] += 1
    return ans

print('sai :', wrong_list(examples[1][0]), '| đúng:', Solution().countPairs(examples[1][0]))

sai : 12 | đúng: 3


**Vì sao sai:** cùng một xâu đích có thể đạt được bằng nhiều dãy phép đổi chỗ (ví dụ đổi rồi đổi lại = xâu gốc) — phải khử trùng bằng `set`.

### Lỗi 4 — Thêm số hiện tại vào bảng **trước** khi đếm

In [16]:
def wrong_order(nums):
    D = len(str(max(nums))); cnt = Counter(); ans = 0
    for x in nums:
        s = str(x).zfill(D)
        cnt[s] += 1                                             # SAI: tự ghép với chính nó
        ans += sum(cnt[t] for t in reach(s))
    return ans

print('sai :', wrong_order(examples[0][0]), '| đúng:', Solution().countPairs(examples[0][0]))

sai : 8 | đúng: 4


**Vì sao sai:** `s ∈ reach(s)` ⇒ mỗi phần tử tự đếm một "cặp" với chính nó.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Đếm cặp thỏa quan hệ đối xứng, mỗi phần tử có ít 'láng giềng' theo quan hệ" ⇒ sinh láng giềng + bảng băm các phần tử đã gặp.
- Thao tác tự nghịch đảo (đổi chỗ) ⇒ thao tác trên cả hai phía quy về một phía.
- So sánh số có số 0 đầu ⇒ đệm về cùng độ dài.

## 15. Biến thể của bài

1. **LC 3265 (bản I):** tối đa một phép.
2. Tối đa `k` phép ⇒ `|reach|` tăng theo `C(D,2)^k`; với `k` lớn, so bằng số chu trình của hoán vị (khoảng cách Cayley).
3. Đếm **nhóm** số đôi một gần bằng (thành phần liên thông) — quan hệ không bắc cầu, cần cẩn thận.
4. Tìm cặp gần bằng có tổng lớn nhất.

In [17]:
# Biến thể 2: khoảng cách Cayley (số phép đổi chỗ tối thiểu) giữa hai xâu cùng đa tập
from functools import lru_cache
def min_swaps(a, b):
    """BFS (xâu ngắn) — số phép đổi chỗ ít nhất biến a thành b."""
    if sorted(a) != sorted(b):
        return None
    from collections import deque
    dist = {a: 0}; q = deque([a])
    while q:
        u = q.popleft()
        if u == b:
            return dist[u]
        s = list(u)
        for i in range(len(s)):
            for j in range(i + 1, len(s)):
                if s[i] != s[j]:
                    s[i], s[j] = s[j], s[i]; v = ''.join(s); s[i], s[j] = s[j], s[i]
                    if v not in dist:
                        dist[v] = dist[u] + 1; q.append(v)

def k_swaps_count(nums, k):
    D = len(str(max(nums))); S = [str(x).zfill(D) for x in nums]
    return sum(1 for i in range(len(S)) for j in range(i + 1, len(S))
               if (d := min_swaps(S[i], S[j])) is not None and d <= k)

rng = random.Random(15)
for _ in range(60):
    nums = [rng.randint(1, 9999) for _ in range(rng.randint(2, 6))]
    assert k_swaps_count(nums, 2) == Solution().countPairs(nums)
    assert k_swaps_count(nums, 1) == wrong_one_swap(nums)
print('Khoảng cách Cayley <= 2 khớp lời giải bản II; <= 1 khớp bản I.')

Khoảng cách Cayley <= 2 khớp lời giải bản II; <= 1 khớp bản I.


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Ghi chú |
|---|---|---|---|
| 1 | BFS theo định nghĩa | rất lớn | Oracle |
| 2 | So từng cặp theo vị trí sai | `O(n² D)` | Không sinh tập |
| 3 | Sinh tập + bảng băm | `O(n D⁴)` | **Nộp bài** |
| 4 | Gom nhóm trùng | `O(u D⁴)` | Tốt khi nhiều số trùng |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `[123, 321, 213]`.

**Bài 2 (tính tay):** `[1000, 1, 10]`.

**Bài 3 (điền TODO):** liệt kê các cặp `(i, j)` gần bằng.

**Bài 4 (tự viết lại):** biến thể 4 — tổng lớn nhất của một cặp gần bằng (hoặc `−1`).

**Bài 5 (mini challenge):** đếm cặp gần bằng với **tối đa 3** phép (dùng khoảng cách Cayley theo chu trình).

In [18]:
def list_pairs_todo(nums):
    # TODO
    raise NotImplementedError

def max_pair_sum_todo(nums):
    # TODO
    raise NotImplementedError

def three_swaps_todo(nums):
    # TODO
    raise NotImplementedError

print('Đã định nghĩa các hàm TODO (không gọi).')

Đã định nghĩa các hàm TODO (không gọi).


## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) `123 ↔ 321`: 1 phép; `123 ↔ 213`: 1 phép; `321 ↔ 213`: chu trình 3 ⇒ 2 phép. (2) Cả ba cặp. (3) `3`.

**Bài 2.** (1) Đệm `1000, 0001, 0010`. (2) Mỗi cặp một phép. (3) `3`.

**Bài 3.** (1) Bảng `chỉ số theo xâu`. (2) Với mỗi `t ∈ reach(s)`, ghép với mọi chỉ số trước. (3) Sắp kết quả.

**Bài 4.** (1) Lưu giá trị lớn nhất theo xâu đã đệm. (2) `x + max{y : pad(y) ∈ reach(pad(x))}`. (3) Vét cạn.

**Bài 5.** (1) Với xâu có chữ số lặp, khoảng cách Cayley không chỉ là `D − #chu trình` (phải chọn cách ghép tốt). (2) Dùng BFS `min_swaps` cho dữ liệu nhỏ. (3) Đối chiếu.

## 19. Đáp án tham khảo cho bài tập

In [19]:
# Bài 1, 2, 3
assert Solution().countPairs([123, 321, 213]) == 3
assert Solution().countPairs([1000, 1, 10]) == 3

def list_pairs(nums):
    D = len(str(max(nums))); where = {}; res = []
    for j, x in enumerate(nums):
        s = str(x).zfill(D)
        for t in reach(s):
            for i in where.get(t, []):
                res.append((i, j))
        where.setdefault(s, []).append(j)
    return sorted(res)

assert list_pairs(examples[0][0]) == [(0, 1), (0, 3), (1, 2), (1, 3)]
print('Bài 1: 3 | Bài 2: 3 | Bài 3:', list_pairs(examples[0][0]))

Bài 1: 3 | Bài 2: 3 | Bài 3: [(0, 1), (0, 3), (1, 2), (1, 3)]


In [20]:
# Bài 4, 5
def max_pair_sum(nums):
    D = len(str(max(nums))); best_of = {}; best = -1
    for x in nums:
        s = str(x).zfill(D)
        for t in reach(s):
            if t in best_of:
                best = max(best, x + best_of[t])
        best_of[s] = max(best_of.get(s, -1), x)
    return best

def three_swaps(nums):
    return k_swaps_count(nums, 3)

rng = random.Random(4)
for _ in range(60):
    nums = [rng.randint(1, 9999) for _ in range(rng.randint(2, 6))]
    exp = max([nums[i] + nums[j] for i in range(len(nums)) for j in range(i + 1, len(nums))
               if almost_equal_def(nums[i], nums[j])], default=-1)
    assert max_pair_sum(nums) == exp
print('Bài 4 khớp định nghĩa. Bài 5 ví dụ [1234, 4123, 2341]:', three_swaps([1234, 4123, 2341]))

Bài 4 khớp định nghĩa. Bài 5 ví dụ [1234, 4123, 2341]: 3


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
D = len(str(max(nums))); cnt = {}
for x in nums:
    s = pad(x, D)
    ans += Σ_{t ∈ reach_≤2(s)} cnt[t]      # reach là SET
    cnt[s] += 1                              # sau khi đếm
```

**Checklist:**

- [ ] Hai phép đổi chỗ (không phải một)?
- [ ] Đệm số 0 về cùng độ dài?
- [ ] Tập `reach` khử trùng?
- [ ] Đếm trước, thêm vào bảng sau?
- [ ] Chu trình độ dài 4 (cần 3 phép) không được tính?